# Transformations and composition

In this notebook, we walk through transformations: steps that prepare a problem for an algorithm and convert the algorithm's output back. We then attach them to an algorithm with [CompositeAlgorithm](https://saminulh.github.io/dpstats/library/reference/algorithm/dpstats.algorithm.CompositeAlgorithm/) and compare compositions in a sweep.

A transformation has two methods:

1. `prepare` receives the metadata, input, privacy budget and seed. It returns the metadata and input for the next step, the budget and seed left for it, and what `restore` needs later.
2. `restore` converts the next step's output back to the original problem.

## Set up an unbounded problem

We draw 1,000 samples in three dimensions from a Gaussian and declare the domain unbounded in every coordinate, as it would be for data with no known range. [compute_metric_arguments](https://saminulh.github.io/dpstats/library/reference/tasks/dpstats.tasks.compute_metric_arguments/) gives the metric arguments, including the sample mean.

In [ ]:
import numpy as np

from dpstats.budgets import ApproxDPBudget
from dpstats.domains import BoxDomain
from dpstats.tasks import MeanEstimation, compute_metric_arguments
from dpstats.tasks.mean_estimation import MeanEstimationInput, MeanEstimationMetadata


def simulate_unbounded(seed, n_samples=1000, dimension=3):
    rng = np.random.default_rng(seed)
    population_mean = rng.normal(scale=5.0, size=dimension)
    X = population_mean + rng.normal(size=(n_samples, dimension))
    metadata = MeanEstimationMetadata(
        n_samples=n_samples,
        domain=BoxDomain(bounds=((-np.inf, np.inf),) * dimension),
    )
    input = MeanEstimationInput(X=X)
    return metadata, input, compute_metric_arguments(MeanEstimation, metadata, input)


metadata, input, arguments = simulate_unbounded(seed=0)
budget = ApproxDPBudget(eps=1.0, delta=1e-6)
print("domain:", metadata.domain)
print("sample mean:", np.round(arguments["sample_mean"], 3))

The [Gaussian Mechanism](https://saminulh.github.io/dpstats/mean_estimation/algorithms/GaussianMechanism/) adds noise scaled to the size of the domain, so it refuses an unbounded domain with `InvalidRegime`.

In [ ]:
from dpstats.exceptions import InvalidRegime
from dpstats.tasks.mean_estimation.algorithms import GaussianMechanism

algorithm = GaussianMechanism()
try:
    algorithm.run(metadata, input, budget, seed=2026)
except InvalidRegime as error:
    print("InvalidRegime:", error)

## Prepare, run, and restore

[ApproxBounds](https://saminulh.github.io/dpstats/mean_estimation/transformations/ApproxBounds/) privately chooses bounds for every unbounded coordinate and clips the data to them. Its `bounds_share` is the fraction of the budget spent on choosing bounds, and the rest passes to the next step. The next cell runs its two methods by hand around the Gaussian Mechanism. Read the output as follows:

1. The prepared domain has finite bounds chosen from the data.
2. The Gaussian Mechanism receives 70% of the budget.
3. `restore` returns the Gaussian Mechanism's estimate unchanged, since clipping does not change the units.

In [ ]:
from dpstats.tasks.mean_estimation.transformations import ApproxBounds

transformation = ApproxBounds(bounds_share=0.3)
prepared_metadata, prepared_input, child_budget, child_seed, restoration = (
    transformation.prepare(metadata, input, budget, seed=2026)
)
child_output = algorithm.run(prepared_metadata, prepared_input, child_budget, child_seed)
restored_output = transformation.restore(child_output, restoration)
print("prepared domain:", prepared_metadata.domain)
print("budget for the Gaussian Mechanism:", child_budget)
print("restored estimate:", np.round(restored_output.mean_est, 3))

## Compose transformations with an algorithm

`CompositeAlgorithm` runs these steps for us: it calls `prepare`, runs the algorithm, and calls `restore`. With the same input, budget and seed, it gives the same estimate as the cell above. The composition spends exactly the offered budget.

In [ ]:
from dpstats.algorithm import CompositeAlgorithm

with_bounds = CompositeAlgorithm(transformations=(ApproxBounds(bounds_share=0.3),), algorithm=algorithm)
output = with_bounds.run(metadata, input, budget, seed=2026)
print(with_bounds)
print("same estimate:", np.allclose(output.mean_est, restored_output.mean_est))

With several transformations:

1. `prepare` runs in the order of the `transformations` tuple.
2. `restore` runs in reverse order.

Order matters. [Standardize](https://saminulh.github.io/dpstats/mean_estimation/transformations/Standardize/) rescales the data using the declared bounds, so it needs finite bounds. Placed after `ApproxBounds`, it receives them; placed first, it refuses the unbounded problem.

In [ ]:
from dpstats.tasks.mean_estimation.transformations import Standardize

bounds_then_standardize = CompositeAlgorithm(
    transformations=(ApproxBounds(bounds_share=0.3), Standardize()), algorithm=algorithm
)
print(bounds_then_standardize, np.round(bounds_then_standardize.run(metadata, input, budget, seed=2026).mean_est, 3))

standardize_then_bounds = CompositeAlgorithm(
    transformations=(Standardize(), ApproxBounds(bounds_share=0.3)), algorithm=algorithm
)
try:
    standardize_then_bounds.run(metadata, input, budget, seed=2026)
except InvalidRegime as error:
    print(standardize_then_bounds, "InvalidRegime:", error)

## Compare compositions in a sweep

The budget spent on bounds is not spent on the estimate, so `bounds_share` is a trade-off. To compare choices, we add three unbounded instances to a local workspace and sweep three compositions that differ only in `bounds_share`, with four privacy budgets and ten seeds each. A `label` names each composition in the results.

In [ ]:
from pathlib import Path

from dpstats import init_workspace, open_collection

collection = open_collection(init_workspace(Path("transformations-workspace")), MeanEstimation)
instances = [
    collection.add_instance(*simulate_unbounded(seed=seed), label=f"unbounded gaussian, seed {seed}")
    for seed in (0, 1, 2)
]
compositions = [
    CompositeAlgorithm(
        label=f"bounds share {share}",
        transformations=(ApproxBounds(bounds_share=share),),
        algorithm=GaussianMechanism(),
    )
    for share in (0.1, 0.3, 0.5)
]
results = collection.run_sweep(
    algorithms=compositions,
    instances=instances,
    budgets=[ApproxDPBudget(eps=eps, delta=1e-6) for eps in (0.3, 1.0, 3.0, 10.0)],
    seeds=range(10),
    time_limit=None,
    memory_limit_gb=None,
)
print("trials:", len(results))

As in the [metrics notebook](https://saminulh.github.io/dpstats/library/tutorials/metrics_and_baselines/), we measure `l2_error_to_sample_mean` and divide it by its statistical baseline, the standard error of the sample mean. The table shows the median ratio over the trials on the three instances. An infinite entry means at least half of those trials released no estimate: with a small share at a small budget, bound selection often fails and the composition returns `no_release`.

In [ ]:
from dpstats.analysis import compute_error_ratio, measure_results
from dpstats.tasks import list_built_in_metrics

metric = list_built_in_metrics(MeanEstimation)["l2_error_to_sample_mean"]
ratios = compute_error_ratio(measure_results(results, metric, collection=collection), metric=metric)
ratios["eps"] = ratios["budget"].map(lambda budget: budget.eps)
ratios.pivot_table(
    index="eps",
    columns="algorithm_label",
    values=metric.name,
    aggfunc="median",
)

The profile curves show, at each threshold $\tau$, the share of trials at $\varepsilon = 1$ whose ratio is at most $\tau$, averaged over the three instances. A trial with no release is within no threshold. Higher curves are better.

In [ ]:
from dpstats.analysis import compute_profile_curves, plot_profiles

at_eps = ratios[ratios["eps"] == 1.0]
curves = compute_profile_curves(at_eps, np.geomspace(0.3, 30.0, 61), metric=metric)
plot_profiles(curves, xlabel="error / standard error of the sample mean").show()

## Adapt a different task

A transformation can also present one task as another. [ToMEstimation](https://saminulh.github.io/dpstats/mean_estimation/transformations/ToMEstimation/) answers a mean estimation problem with an M-estimation algorithm: the mean is the point that minimizes the squared error to the records, so it passes the records through unchanged and reports the minimizing point as the mean. Its `loss` parameter swaps in another loss. It spends none of the budget.

The parameter domain it declares is the smallest convex set containing the data domain, so it needs finite bounds. We place it after `ApproxBounds` and answer the problem with [DPSGD](https://saminulh.github.io/dpstats/m_estimation/algorithms/DPSGD/), private gradient descent.

In [ ]:
from dpstats.tasks.m_estimation.algorithms import DPSGD
from dpstats.tasks.mean_estimation.transformations import ToMEstimation

adapted = CompositeAlgorithm(
    transformations=(ApproxBounds(bounds_share=0.3), ToMEstimation()),
    algorithm=DPSGD(),
)
adapted_output = adapted.run(metadata, input, budget, seed=2026)
print(adapted)
print("estimate:", np.round(adapted_output.mean_est, 3))
print("sample mean:", np.round(arguments["sample_mean"], 3))

## Next steps

[Compare on the benchmark](compare_on_the_benchmark.ipynb) wraps a method in a task adapter, private bounds and standardization, and compares it on the benchmark. The [transformation reference](https://saminulh.github.io/dpstats/library/reference/algorithm/#transformations) lists the available transformations and their requirements.